In [1]:
import folium
import polars as pl
import mcr_py

In [ ]:
bounding_box = [
    (6.912789559592028, 50.95082420629814),
    (6.90965014627875, 50.94788724437913),
    (6.912218504273028, 50.94473937866948),
    (6.914029522915655, 50.944306400295005),
    (6.916940125566271, 50.946735585299706),
    (6.919886618643858, 50.94555227853371),
    (6.9237266503584465, 50.94991721673625),
    (6.926592441418052, 50.951705692912014),
    (6.925031390065072, 50.95226859486215),
    (6.923008239592917, 50.95206669530157),
    (6.923878775814018, 50.952914472192305),
    (6.921391875103581, 50.95386602541586),
    (6.919528164440663, 50.951875256766186),
    (6.912789559592028, 50.95082420629814),
]

(df_nodes, df_edges, _) = mcr_py.load_osm_walking(
    "koeln", bounding_box, "../osmtools/data/", "../osmtools/data", False
)
(df_c_nodes, df_c_edges, _) = mcr_py.load_osm_cycling(
    "koeln", bounding_box, True, "../osmtools/data/", "../osmtools/data", False
)
(df_d_nodes, df_d_edges, _) = mcr_py.load_osm_driving(
    "koeln", bounding_box, "../osmtools/data/", "../osmtools/data", False
)
(df_pois) = mcr_py.load_osm_pois(
    "koeln",
    bounding_box,
    "../osmtools/data/",
    "../osmtools/data",
    False,
    nodes_to_match_df=df_nodes,  # "../osmtools/data/koeln_walking_nodes.parquet",
)

In [2]:
import mcr_py.osm.graph

df_nodes = pl.read_parquet("../osmtools/data/koeln_walking_nodes.parquet")
df_c_nodes = pl.read_parquet("../osmtools/data/koeln_cycling_nodes.parquet")
df_d_nodes = pl.read_parquet("../osmtools/data/koeln_driving_nodes.parquet")
df_edges = pl.read_parquet("../osmtools/data/koeln_walking_edges.parquet")
df_c_edges = pl.read_parquet("../osmtools/data/koeln_cycling_edges.parquet")
df_d_edges = pl.read_parquet("../osmtools/data/koeln_driving_edges.parquet")
df_pois = pl.read_parquet("../osmtools/data/koeln_pois_nodes.parquet")
(df_nodes, df_edges, graph) = mcr_py.osm.graph.create_rx_graph(df_nodes, df_edges)

In [3]:
df_pois.columns

['osm_id', 'lat', 'long', 'nearest_osm_node', 'dist_to_nearest']

In [4]:
(graph_after, nodes_after, edges_after) = (
    mcr_py.osm.graph.crop_graph_to_largest_component(graph, df_nodes, df_edges)
)
paths = mcr_py.osm.graph.shortest_paths(graph_after)
df_pois_added = mcr_py.add_nearest_node_to_df(df_pois, nodes_after, "EPSG:4839")

In [5]:
df_pois_added

osm_id,lat,long,nearest_osm_node,dist_to_nearest,nearest_node_osm_id,nearest_node_distance
u64,f64,f64,u64,f64,u64,f64
196190188,50.951854,6.9246184,1705497069,163.879259,1705497069,276.064244
196191317,50.951599,6.9252478,8807730998,80.181867,8807730998,226.851236
243914656,50.952831,6.9224452,1705497086,47.82041,9683627914,27138.269898
243914657,50.952767,6.9226077,1705497086,78.614088,9683627914,23629.851663
243919932,50.952213,6.9206586,3470214886,126.110814,295482209,5327.173936
…,…,…,…,…,…,…
12140200351,50.950722,6.9130704,445516246,122.132194,445516224,2586.965301
12216279733,50.953444,6.9216243,1705497097,123.132947,32248902,71920.504377
12269097381,50.949785,6.9120492,3456426120,786.396828,3456426120,2766.238911


In [7]:
df_pois = df_pois.join(
    df_nodes.select(
        pl.col("osm_id"),
        pl.col("lat").alias("nearest_lat"),
        pl.col("long").alias("nearest_lon"),
    ),
    left_on="nearest_osm_node",
    right_on="osm_id",
)


def get_coordinates(df_edges, df_nodes):
    return df_edges.join(
        df_nodes.select(
            pl.col("osm_id"),
            pl.col("lat").alias("from_lat"),
            pl.col("long").alias("from_lon"),
        ),
        left_on="source_osm",
        right_on="osm_id",
    ).join(
        df_nodes.select(
            pl.col("osm_id"),
            pl.col("lat").alias("dest_lat"),
            pl.col("long").alias("dest_lon"),
        ),
        left_on="dest_osm",
        right_on="osm_id",
    )


df_edges = get_coordinates(df_edges, df_nodes)
df_c_edges = get_coordinates(df_c_edges, df_c_nodes)
df_d_edges = get_coordinates(df_d_edges, df_d_nodes)
edges_after = get_coordinates(edges_after, nodes_after)

In [8]:
df_edges = df_edges.with_columns(
    pl.concat_list(pl.col("source").cast(pl.String), pl.col("dest").cast(pl.String))
    .list.sort()
    .list.join("")
    .is_duplicated()
    .alias("duplicated")
)
df_c_edges = df_c_edges.with_columns(
    pl.concat_list(pl.col("source").cast(pl.String), pl.col("dest").cast(pl.String))
    .list.sort()
    .list.join("")
    .is_duplicated()
    .alias("duplicated")
)
df_d_edges = df_d_edges.with_columns(
    pl.concat_list(pl.col("source").cast(pl.String), pl.col("dest").cast(pl.String))
    .list.sort()
    .list.join("")
    .is_duplicated()
    .alias("duplicated")
)

In [17]:
m = folium.Map(location=[50.949, 6.916], zoom_start=15)
fg_walking = folium.FeatureGroup("Walking")
for edge in df_edges.iter_rows(named=True):
    folium.PolyLine(
        locations=[
            [edge["from_lat"], edge["from_lon"]],
            [edge["dest_lat"], edge["dest_lon"]],
        ],
        color="blue" if not edge["duplicated"] else "lightblue",
        popup=f"{edge['source_osm']}",
        weight=2,
        opacity=1,
    ).add_to(fg_walking)
fg_walking.add_to(m)

fg_cycling = folium.FeatureGroup("Cycling")
for edge in df_c_edges.iter_rows(named=True):
    folium.PolyLine(
        locations=[
            [edge["from_lat"], edge["from_lon"]],
            [edge["dest_lat"], edge["dest_lon"]],
        ],
        color="green" if not edge["duplicated"] else "lightgreen",
        weight=2,
        opacity=1,
    ).add_to(fg_cycling)
fg_cycling.add_to(m)
fg_driving = folium.FeatureGroup("Driving")
for edge in df_d_edges.iter_rows(named=True):
    folium.PolyLine(
        locations=[
            [edge["from_lat"], edge["from_lon"]],
            [edge["dest_lat"], edge["dest_lon"]],
        ],
        color="red" if not edge["duplicated"] else "lightcoral",
        weight=2,
        opacity=1,
    ).add_to(fg_driving)
fg_driving.add_to(m)
fg_cropped = folium.FeatureGroup("Cropped")
for edge in edges_after.iter_rows(named=True):
    folium.PolyLine(
        locations=[
            [edge["from_lat"], edge["from_lon"]],
            [edge["dest_lat"], edge["dest_lon"]],
        ],
        color="brown",
        weight=2,
        opacity=1,
    ).add_to(fg_cropped)
fg_cropped.add_to(m)

for node in nodes_after.head().iter_rows(named=True):
    folium.Marker(
        location=[node["lat"], node["long"]], popup=node["rx_node_id"]
    ).add_to(fg_cropped)


folium.LayerControl().add_to(m)
m

In [ ]:
m = folium.Map(location=[50.949, 6.916], zoom_start=15)
for node in df_pois.iter_rows(named=True):
    folium.Circle(
        location=[node["lat"], node["long"]],
        popup=f"Node {node['osm_id']}",
        radius=1,
        color="green",
    ).add_to(m)

    folium.Circle(
        location=[node["nearest_lat"], node["nearest_lon"]],
        popup=f"Node {node['nearest_osm_node']}",
        radius=1,
    ).add_to(m)

    folium.PolyLine(
        locations=[
            [node["lat"], node["long"]],
            [node["nearest_lat"], node["nearest_lon"]],
        ],
        color="red",
        weight=2,
        opacity=1,
    ).add_to(m)

for node in df_nodes.iter_rows(named=True):
    folium.Circle(
        location=[node["lat"], node["long"]],
        popup=f"Node {node['osm_id']}",
        radius=1,
    ).add_to(m)

m